In [24]:
import numpy as np
import pandas as pd
import pyvista as pv
from PIL import Image
import plotly.io as pio
import plotly.express as px
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from sklearn.decomposition import PCA

In [25]:
pd.set_option('display.max_columns', None)

In [26]:
df = pd.read_csv('champions_db.csv')

In [27]:
df.head(3)

,name,image,tags,hp,hpperlevel,mp,mpperlevel,movespeed,armor,armorperlevel,spellblock,spellblockperlevel,attackrange,hpregen,hpregenperlevel,mpregen,mpregenperlevel,crit,critperlevel,attackdamage,attackdamageperlevel,attackspeed,attackspeedperlevel,image_path,image_url
0,Aatrox,Aatrox.png,"['Fighter', 'Tank']",650,114,0,0.0,345,38,4.45,32,2.05,175,3.0,1.0,0.0,0.0,0,0,60,5.0,0.651,2.5,assets/LoL/Aatrox.png,http://ddragon.leagueoflegends.com/cdn/14.8.1/...
1,Ahri,Ahri.png,"['Mage', 'Assassin']",590,104,418,25.0,330,21,4.70,30,1.30,550,2.5,0.6,8.0,0.8,0,0,53,3.0,0.668,2.2,assets/LoL/Ahri.png,http://ddragon.leagueoflegends.com/cdn/14.8.1/...
2,Akali,Akali.png,['Assassin'],600,119,200,0.0,345,23,4.70,37,2.05,125,9.0,0.9,50.0,0.0,0,0,62,3.3,0.625,3.2,assets/LoL/Akali.png,http://ddragon.leagueoflegends.com/cdn/14.8.1/...


In [28]:
# O mp do Viego vem 10_000, arrumando para 0 para não dar outlier
df.loc[df['name'] == 'Viego', 'mp'] = 0

# O mp da Bel'Veth vem 480, arrumando para 0 para não dar outlier
df.loc[df['name'] == "Bel'Veth", 'mp'] = 0

### PCA 2D

In [29]:
df[['hp', 'mp', 'movespeed', 'armor', 'spellblock', 'attackrange', 'attackspeed', 'attackdamage']][df['name'].isin(['Aatrox', 'Viego', """Bel'Veth""", 'Garen', 'Zed', 'Ziggs', 'Graves'])]

,hp,mp,movespeed,armor,spellblock,attackrange,attackspeed,attackdamage
0,650,0,345,38,32,175,0.651,60
13,610,0,340,32,32,175,0.850,60
36,690,0,340,38,32,175,0.625,69
39,625,325,340,33,32,425,0.475,68
148,630,0,345,34,32,200,0.658,57
161,654,200,345,32,29,125,0.651,63
163,606,480,325,21,30,550,0.656,55


In [30]:
pca_2d = PCA(n_components=2)
df_pca2d = pd.DataFrame(
    pca_2d.fit_transform(
        # df_scaled
        df[['hp', 'mp', 'movespeed', 'armor', 'spellblock', 'attackrange', 'attackspeed', 'attackdamage']]
    ),
    columns=['PC1', 'PC2']
)

In [31]:
explained_variance = pca_2d.explained_variance_
explained_variance_ratio = pca_2d.explained_variance_ratio_
print(explained_variance, explained_variance_ratio, np.cumsum(pca_2d.explained_variance_ratio_))

[41632.97595184 11221.71435889] [0.76817958 0.20705442] [0.76817958 0.975234  ]


In [32]:
df_pca2d['champion'] = df['name'].values
df_pca2d['image'] = df['image_path'].values

In [33]:
df_pca2d

,PC1,PC2,champion,image
0,-257.305906,-236.945595,Aatrox,assets/LoL/Aatrox.png
1,246.613405,17.832259,Ahri,assets/LoL/Ahri.png
2,-227.614724,-34.061234,Akali,assets/LoL/Akali.png
3,172.251530,-26.535159,Akshan,assets/LoL/Akshan.png
4,-181.639676,108.854698,Alistar,assets/LoL/Alistar.png
...,...,...,...,...
162,139.112219,-120.918635,Zeri,assets/LoL/Zeri.png
163,267.441811,76.253910,Ziggs,assets/LoL/Ziggs.png
164,260.014171,49.078756,Zilean,assets/LoL/Zilean.png
165,245.294214,25.763029,Zoe,assets/LoL/Zoe.png


In [34]:
import nbformat
print(nbformat.__version__)

5.10.4


In [46]:
pio.renderers.default = "vscode"

fig = go.Figure()

for _, row in df_pca2d.iterrows():
    fig.add_layout_image(
        dict(
            source=row["image"],
            x=row["PC1"],
            y=row["PC2"],
            xref="x",
            yref="y",
            sizex=55,
            sizey=55,
            xanchor="center",
            yanchor="middle",
            layer="above"
        )
    )

fig.update_xaxes(title="PC1")
fig.update_yaxes(title="PC2")

fig.update_layout(
    xaxis=dict(range=[df_pca2d["PC1"].min()-25, df_pca2d["PC1"].max()+25]),
    yaxis=dict(range=[df_pca2d["PC2"].min()-25, df_pca2d["PC2"].max()+25]),
    height=800
)

fig.show()

### PCA 3D

In [36]:
pca_3d = PCA(n_components=3)
df_pca3d = pd.DataFrame(
    pca_3d.fit_transform(
        # df_scaled
        df[['hp', 'mp', 'movespeed', 'armor', 'spellblock', 'attackrange', 'attackspeed', 'attackdamage']]
    ),
    columns=['PC1', 'PC2', 'PC3']
)

In [37]:
explained_variance = pca_3d.explained_variance_
explained_variance_ratio = pca_3d.explained_variance_ratio_
print(explained_variance, explained_variance_ratio, np.cumsum(pca_3d.explained_variance_ratio_))

[41632.97595184 11221.71435889  1271.57746689] [0.76817958 0.20705442 0.02346217] [0.76817958 0.975234   0.99869617]


In [38]:
df_pca3d['champion'] = df['name'].values
df_pca3d['image'] = df['image_path'].values

In [39]:
df_pca3d

,PC1,PC2,PC3,champion,image
0,-257.305906,-236.945595,17.033534,Aatrox,assets/LoL/Aatrox.png
1,246.613405,17.832259,-7.459515,Ahri,assets/LoL/Ahri.png
2,-227.614724,-34.061234,-37.683827,Akali,assets/LoL/Akali.png
3,172.251530,-26.535159,27.677539,Akshan,assets/LoL/Akshan.png
4,-181.639676,108.854698,47.671750,Alistar,assets/LoL/Alistar.png
...,...,...,...,...,...
162,139.112219,-120.918635,-2.172689,Zeri,assets/LoL/Zeri.png
163,267.441811,76.253910,8.569093,Ziggs,assets/LoL/Ziggs.png
164,260.014171,49.078756,-23.341816,Zilean,assets/LoL/Zilean.png
165,245.294214,25.763029,32.376264,Zoe,assets/LoL/Zoe.png


In [51]:
plotter = pv.Plotter()
# def format_name(name):
#     return name.replace("'", "").replace(" ", "")

for _, row in df_pca3d.iterrows():
    x, y, z = row['PC1'], row['PC2'], row['PC3']
    
    try:
        img = Image.open(row['image'])
        texture = pv.Texture(np.array(img))
        
        plane = pv.Plane(center=(x, y, z), i_size=40, j_size=40)

        plotter.add_mesh(plane, texture=texture)
        plotter.add_axes()

        axis_length = 500

        x_axis = pv.Line(pointa=(-axis_length, 0, 0), pointb=(axis_length, 0, 0))
        plotter.add_mesh(x_axis, color='red', line_width=3)

        y_axis = pv.Line(pointa=(0, -axis_length, 0), pointb=(0, axis_length, 0))
        plotter.add_mesh(y_axis, color='green', line_width=3)

        z_axis = pv.Line(pointa=(0, 0, -axis_length), pointb=(0, 0, axis_length))
        plotter.add_mesh(z_axis, color='blue', line_width=3)

    except:
        plotter.add_mesh(pv.Sphere(radius=0.1, center=(x, y, z)))

plotter.show()

Widget(value='<iframe src="http://localhost:60872/index.html?ui=P_0x284130a4260_8&reconnect=auto" class="pyvis…

In [ ]:
# # pio.renderers.default = "browser"
# pio.renderers.default = "vscode"

# fig = px.scatter_3d(
#     df_pca3d,
#     x='PC1',
#     y='PC2',
#     z='PC3',
#     hover_name='champion',
#     custom_data=['img']
# )

# # cria lista de campeões
# champions = df_pca3d['champion'].unique()

# # cria botões
# buttons = []

# for champ in champions:
#     visible = [c == champ for c in df_pca3d['champion']]

#     buttons.append(
#         dict(
#             label=champ,
#             method="update",
#             args=[{
#                 "marker": {
#                     "size": [20 if v else 3 for v in visible],
#                     "opacity": [1 if v else 0.2 for v in visible]
#                 }
#             }]
#         )
#     )


# fig.show()

ValueError: Value of 'custom_data_0' is not the name of a column in 'data_frame'. Expected one of ['PC1', 'PC2', 'PC3', 'champion', 'image'] but received: img